## AI SUPPORT TICKET CLASSIFIER - DATA EXPLORATION & PREPARATION
This notebook builds an AI system to classify support tickets by priority and route them to the correct department using Groq API

### Step 1: Import all required libraries

In [1]:
import pandas as pd                    # Data manipulation (like Excel)
import numpy as np                     # Numerical operations
import matplotlib.pyplot as plt        # Create charts
import seaborn as sns                  # Beautiful statistical plots
from collections import Counter       # Count occurrences
import warnings                        # Handle warnings
import json
from groq import Groq
import os
from dotenv import load_dotenv

# Don't show warnings (cleaner output)
warnings.filterwarnings('ignore')

# Set style for plots (makes them look nice)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ All libraries imported successfully!")
print("\nLibraries loaded:")
print("  - pandas: For data manipulation")
print("  - numpy: For numerical operations")
print("  - matplotlib: For creating visualizations")
print("  - seaborn: For statistical plots")

✅ All libraries imported successfully!

Libraries loaded:
  - pandas: For data manipulation
  - numpy: For numerical operations
  - matplotlib: For creating visualizations
  - seaborn: For statistical plots


### STEP 2: LOAD THE DATASET

In [2]:
file_path = 'C:/Users/user/Desktop/YIV/Project AI/aa_dataset-tickets-multi-lang-5-2-50-version.csv'

# Load the CSV file into a pandas DataFrame
# pd.read_csv() = "pandas, read this CSV file"
# The result is stored in a variable called 'df' (short for DataFrame)

df = pd.read_csv(file_path)

# Print basic information
print("=" * 80)
print("DATASET LOADED SUCCESSFULLY!")
print("=" * 80)
print(f"\n Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"   - {df.shape[0]} support tickets")
print(f"   - {df.shape[1]} pieces of information per ticket")

print("\n" + "=" * 80)
print("COLUMN NAMES (What information we have):")
print("=" * 80)
for i, col in enumerate(df.columns, 1):
    print(f"{i:2d}. {col}")

DATASET LOADED SUCCESSFULLY!

 Dataset Shape: 28587 rows, 16 columns
   - 28587 support tickets
   - 16 pieces of information per ticket

COLUMN NAMES (What information we have):
 1. subject
 2. body
 3. answer
 4. type
 5. queue
 6. priority
 7. language
 8. version
 9. tag_1
10. tag_2
11. tag_3
12. tag_4
13. tag_5
14. tag_6
15. tag_7
16. tag_8


### STEP 3: EXPLORE THE DATASET

In [3]:
print("=" * 80)
print("FIRST 3 TICKETS (What actual data looks like):")
print("=" * 80)

# Display first 3 tickets (rows)
for idx, row in df.head(3).iterrows():
    print(f"\n- TICKET {idx + 1}:")
    print(f"   Subject: {row['subject'][:80]}...")  # First 80 characters
    print(f"   Language: {row['language']}")
    print(f"   Type: {row['type']}")
    print(f"   Priority: {row['priority']}")
    print(f"   Queue: {row['queue']}")
    print(f"   Body preview: {row['body'][:100]}...")  # First 100 characters

print("\n" + "=" * 80)
print("DATA TYPES (What kind of data each column contains):")
print("=" * 80)
print(df.dtypes)

print("\n" + "=" * 80)
print("MISSING VALUES (Empty cells in each column):")
print("=" * 80)
missing = df.isna().sum()
for col, count in missing.items():
    percentage = (count / len(df)) * 100
    if count > 0:
        print(f"   {col}: {count} missing ({percentage:.1f}%)")
    else:
        print(f"   {col}: Complete")

FIRST 3 TICKETS (What actual data looks like):

- TICKET 1:
   Subject: Wesentlicher Sicherheitsvorfall...
   Language: de
   Type: Incident
   Priority: high
   Queue: Technical Support
   Body preview: Sehr geehrtes Support-Team,\n\nich möchte einen gravierenden Sicherheitsvorfall melden, der gegenwär...

- TICKET 2:
   Subject: Account Disruption...
   Language: en
   Type: Incident
   Priority: high
   Queue: Technical Support
   Body preview: Dear Customer Support Team,\n\nI am writing to report a significant problem with the centralized acc...

- TICKET 3:
   Subject: Query About Smart Home System Integration Features...
   Language: en
   Type: Request
   Priority: medium
   Queue: Returns and Exchanges
   Body preview: Dear Customer Support Team,\n\nI hope this message reaches you well. I am reaching out to request de...

DATA TYPES (What kind of data each column contains):
subject       str
body          str
answer        str
type          str
queue         str
priority      s

### STEP 4: ANALYZE PRIORITY DISTRIBUTION

In [4]:
print("=" * 80)
print("PRIORITY DISTRIBUTION (How many tickets in each priority?)")
print("=" * 80)

# Count tickets by priority
priority_counts = df['priority'].value_counts()

print("\nAbsolute counts:")
for priority, count in priority_counts.items():
    percentage = (count / len(df)) * 100
    bar = "█" * int(percentage / 2)
    print(f"  {priority:8s}: {count:6d} tickets ({percentage:5.1f}%) {bar}")

print("\n" + "=" * 80)
print("MEANING:")
print("=" * 80)
print("""
This shows us the class balance:
- If a priority level is very rare (e.g., 2%), AI will struggle to learn it
- If all tickets are "low", AI can just predict "low" for everything
- Balanced data (30/30/40%) is better for learning

For us: We need AI to be good at predicting ALL priority levels, not just common ones.
""")

# Store priority distribution for later
priority_distribution = priority_counts

PRIORITY DISTRIBUTION (How many tickets in each priority?)

Absolute counts:
  medium  :  11515 tickets ( 40.3%) ████████████████████
  high    :  11178 tickets ( 39.1%) ███████████████████
  low     :   5894 tickets ( 20.6%) ██████████

MEANING:

This shows us the class balance:
- If a priority level is very rare (e.g., 2%), AI will struggle to learn it
- If all tickets are "low", AI can just predict "low" for everything
- Balanced data (30/30/40%) is better for learning

For us: We need AI to be good at predicting ALL priority levels, not just common ones.



### STEP 5: ANALYZE TEXT DATA (What AI will read)

In [5]:
print("=" * 80)
print("TEXT LENGTH ANALYSIS (How long are tickets?)")
print("=" * 80)

# Calculate length of body text for each ticket
df['body_length'] = df['body'].str.len()

print(f"\nAverage ticket length: {df['body_length'].mean():.0f} characters")
print(f"Shortest ticket: {df['body_length'].min()} characters")
print(f"Longest ticket: {df['body_length'].max()} characters")

print("\n" + "=" * 80)
print("SAMPLE TICKETS BY PRIORITY (How they differ)")
print("=" * 80)

for priority in ['low', 'medium', 'high']:
    print(f"\n{'='*80}")
    print(f"SAMPLE {priority.upper()} PRIORITY TICKET:")
    print(f"{'='*80}")
    
    # Get one random ticket with this priority
    sample = df[df['priority'] == priority].sample(1).iloc[0]
    
    print(f"Subject: {sample['subject']}")
    print(f"Language: {sample['language']}")
    print(f"Queue: {sample['queue']}")
    print(f"Type: {sample['type']}")
    print(f"\nBody (full text):")
    print(f"{sample['body']}\n")
    print(f"Agent's Response (how it was solved):")
    print(f"{sample['answer'][:300]}...")

TEXT LENGTH ANALYSIS (How long are tickets?)

Average ticket length: 387 characters
Shortest ticket: 6 characters
Longest ticket: 1469 characters

SAMPLE TICKETS BY PRIORITY (How they differ)

SAMPLE LOW PRIORITY TICKET:
Subject: Sicherung medizinischer Daten auf WordPress für Gesundheitsorganisationen
Language: de
Queue: Customer Service
Type: Request

Body (full text):
Sehr geehrte Kundenservice, ich schreibe an, um Rat zur Sicherung medizinischer Daten innerhalb einer WordPress-Website für Gesundheitsorganisationen einzuholen. Da Gesundheitsorganisationen sensible Patientendaten verwalten, müssen sie sicherstellen, dass ihre Website mit den relevanten Vorschriften ins Einklang steht. Könnten Sie Empfehlungen zu WordPress-Plugins und Themes geben, die den besten Praktiken entsprechen und helfen, eine WordPress-Website sicherzustellen? Ich würde sehr schätzen, wenn Sie Empfehlungen zu Datenaufbewahrung, Zugangssteuerungen und Rückup-Verfahren geben. Darüber hinaus bin ich an spezifisc

### STEP 6: DATA QUALITY CHECK & CLEANING

In [6]:
print("=" * 80)
print("DATA QUALITY ASSESSMENT")
print("=" * 80)

# Check for missing values in critical columns
critical_columns = ['subject', 'body', 'priority', 'queue', 'type', 'language']

print("\nMissing values in critical columns:")
for col in critical_columns:
    missing_count = df[col].isna().sum()
    missing_pct = (missing_count / len(df)) * 100
    status = " OK" if missing_count == 0 else "  NEEDS CLEANING"
    print(f"  {col:15s}: {missing_count:5d} missing ({missing_pct:5.1f}%) {status}")

print("\n" + "=" * 80)
print("CLEANING STEP 1: Remove tickets with missing text")
print("=" * 80)

# Count before cleaning
before = len(df)
print(f"Before cleaning: {before} tickets")

# Remove rows where subject OR body is missing
# We need at least subject or body to process the ticket
df_clean = df.dropna(subset=['body'])  # Keep only rows with body text

after = len(df_clean)
removed = before - after
print(f"After cleaning: {after} tickets")
print(f"Removed: {removed} tickets ({(removed/before)*100:.2f}%)")

print("\n" + "=" * 80)
print("CLEANING STEP 2: Fill missing subject with 'No subject'")
print("=" * 80)

# If subject is missing, replace with a placeholder
df_clean['subject'] = df_clean['subject'].fillna('No subject provided')
print(f" Missing subjects filled: {df_clean['subject'].isna().sum()} remaining")

print("\n" + "=" * 80)
print("CLEANING STEP 3: Check for invalid priority values")
print("=" * 80)

# Check what priority values we have
print("Valid priority values in data:")
print(df_clean['priority'].unique())

print("\n" + "=" * 80)
print("SUMMARY:")
print("=" * 80)
print(f" Dataset is now clean!")
print(f" Ready for AI processing")
print(f" Working with {len(df_clean)} tickets")

# Update our main dataframe
df = df_clean

DATA QUALITY ASSESSMENT

Missing values in critical columns:
  subject        :  3838 missing ( 13.4%)   NEEDS CLEANING
  body           :     0 missing (  0.0%)  OK
  priority       :     0 missing (  0.0%)  OK
  queue          :     0 missing (  0.0%)  OK
  type           :     0 missing (  0.0%)  OK
  language       :     0 missing (  0.0%)  OK

CLEANING STEP 1: Remove tickets with missing text
Before cleaning: 28587 tickets
After cleaning: 28587 tickets
Removed: 0 tickets (0.00%)

CLEANING STEP 2: Fill missing subject with 'No subject'
 Missing subjects filled: 0 remaining

CLEANING STEP 3: Check for invalid priority values
Valid priority values in data:
<ArrowStringArray>
['high', 'medium', 'low']
Length: 3, dtype: str

SUMMARY:
 Dataset is now clean!
 Ready for AI processing
 Working with 28587 tickets


### STEP 7: FINAL DATA SUMMARY

In [7]:
print("DATA PREPARATION COMPLETE!")
print("\n" + "=" * 80)
print("FINAL DATASET SUMMARY")
print("=" * 80)

summary = {
    "Total Tickets": len(df),
    "Columns": len(df.columns),
    "Languages": df['language'].nunique(),
    "Ticket Types": df['type'].nunique(),
    "Priority Levels": df['priority'].nunique(),
    "Departments/Queues": df['queue'].nunique(),
}

for key, value in summary.items():
    print(f"  {key:20s}: {value}")

print("\n" + "=" * 80)
print("PRIORITY BREAKDOWN (What we'll predict):")
print("=" * 80)
for priority in sorted(df['priority'].unique()):
    count = (df['priority'] == priority).sum()
    pct = (count / len(df)) * 100
    print(f"  {priority:8s}: {count:6d} tickets ({pct:5.1f}%)")

print("\n" + "=" * 80)
print("WHAT WE HAVE LEARNED:")
print("=" * 80)
print(f"""
 We have {len(df):,} real customer support tickets
 Data is MULTILINGUAL: German and English
 Covers multiple departments: Technical Support, Billing, Sales, etc.
 3 priority levels: low, medium, high (balanced distribution)
 Rich text data: Customer descriptions + Agent responses
 Data quality: Clean and ready for AI processing

""")

print("=" * 80)

DATA PREPARATION COMPLETE!

FINAL DATASET SUMMARY
  Total Tickets       : 28587
  Columns             : 17
  Languages           : 2
  Ticket Types        : 4
  Priority Levels     : 3
  Departments/Queues  : 10

PRIORITY BREAKDOWN (What we'll predict):
  high    :  11178 tickets ( 39.1%)
  low     :   5894 tickets ( 20.6%)
  medium  :  11515 tickets ( 40.3%)

WHAT WE HAVE LEARNED:

 We have 28,587 real customer support tickets
 Data is MULTILINGUAL: German and English
 Covers multiple departments: Technical Support, Billing, Sales, etc.
 3 priority levels: low, medium, high (balanced distribution)
 Rich text data: Customer descriptions + Agent responses
 Data quality: Clean and ready for AI processing


